##Baseline Method

In [ ]:
import os, cv2, torch, numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.utils as vutils
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn.metrics import mutual_info_score

# CONFIGURATION

BASE_PATH = "/kaggle/input/datasets/mdrafsanyeasir/medical-dataset45-thesis/final_preprocessed_dataset"
MODALITY = "CT-MRI"       
EPOCHS =5
BATCH_SIZE = 4
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# DATASET CLASS

class FusionDataset(Dataset):
    def __init__(self, base_path, split, modality):
        mod1, mod2 = modality.split("-")
        self.path1 = os.path.join(base_path, split, modality, mod1)
        self.path2 = os.path.join(base_path, split, modality, mod2)
        self.imgs = sorted(os.listdir(self.path1))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img1 = cv2.imread(os.path.join(self.path1, self.imgs[idx]))
        img2 = cv2.imread(os.path.join(self.path2, self.imgs[idx]))

        img1 = cv2.resize(img1, (256,256)) / 255.0
        img2 = cv2.resize(img2, (256,256)) / 255.0

        img1 = torch.tensor(img1).permute(2,0,1).float()
        img2 = torch.tensor(img2).permute(2,0,1).float()

        return img1, img2


# MODEL ARCHITECTURE

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self,x):
        return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128,64,2,stride=2), nn.ReLU(),
            nn.ConvTranspose2d(64,3,2,stride=2),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.net(x)

class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x1, x2):
        f1 = self.encoder(x1)
        f2 = self.encoder(x2)
        fused = (f1 + f2) / 2   # Average fusion rule
        return self.decoder(fused)


# TRAINING

train_dataset = FusionDataset(BASE_PATH, "train", MODALITY)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model = FusionNet().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Training Started...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for img1, img2 in train_loader:
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)

        fused = model(img1, img2)
        loss = criterion(fused, img1) + criterion(fused, img2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss: {total_loss/len(train_loader):.4f}")


# SAVE FUSED IMAGES (TEST)

os.makedirs(f"fused_outputs/{MODALITY}", exist_ok=True)

test_dataset = FusionDataset(BASE_PATH, "test", MODALITY)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model.eval()
print("Generating fused images...")
with torch.no_grad():
    for i, (img1, img2) in enumerate(test_loader):
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
        fused = model(img1, img2)
        vutils.save_image(fused, f"fused_outputs/{MODALITY}/fused_{i}.png")


# EVALUATION METRICS (FIXED)

def compute_metrics(img1, img2, fused):
    img1 = img1.cpu().numpy().transpose(1,2,0)
    img2 = img2.cpu().numpy().transpose(1,2,0)
    fused = fused.cpu().numpy().transpose(1,2,0)

    psnr = (peak_signal_noise_ratio(img1, fused, data_range=1.0) +
            peak_signal_noise_ratio(img2, fused, data_range=1.0)) / 2

    ssim = (structural_similarity(img1, fused, channel_axis=2, data_range=1.0) +
            structural_similarity(img2, fused, channel_axis=2, data_range=1.0)) / 2

    mi = (mutual_info_score(img1[:,:,0].ravel(), fused[:,:,0].ravel()) +
          mutual_info_score(img2[:,:,0].ravel(), fused[:,:,0].ravel())) / 2

    return psnr, ssim, mi


# METRIC CALCULATION

psnr_list, ssim_list, mi_list = [], [], []

with torch.no_grad():
    for img1, img2 in test_loader:
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
        fused = model(img1, img2)

        psnr, ssim, mi = compute_metrics(img1[0], img2[0], fused[0])
        psnr_list.append(psnr)
        ssim_list.append(ssim)
        mi_list.append(mi)

print("\nFINAL RESULTS")
print("Average PSNR:", np.mean(psnr_list))
print("Average SSIM:", np.mean(ssim_list))
print("Average MI  :", np.mean(mi_list))